# Train a real GuildLM Go specialist for **$0** on Kaggle

This notebook trains the **first GuildLM Code Guild Go specialist** — a QLoRA
adapter — on the **teacher-distilled dataset (146 compile-verified Go examples)**,
using [`anvil`](https://github.com/guildlm/anvil), the GuildLM training forge, on
Kaggle's **free** T4 GPU. **No dataset upload needed** — the dataset is committed
in `guild-code`, which this notebook clones for you.

**The free stack:** Kaggle (free GPU training) → HuggingFace Hub (free hosting) → Ollama (local serving).

**What it does, top to bottom:**
1. Install anvil's training stack + clone the `guild-code` recipe & dataset.
2. Load the committed **teacher** Go SFT dataset with anvil's **real** `src.data` loaders.
3. Train a QLoRA LoRA adapter with anvil's **real** `src.train.train()` entrypoint.
4. Save the adapter to `/kaggle/working/go_reviewer_adapter`.
5. Quick sanity generation with the trained adapter.
6. *(Optional)* push the adapter to the HuggingFace Hub with `anvil-push`.

**Expected runtime:** this is a **real** run — `Qwen2.5-Coder-7B` 4-bit (QLoRA) on
the 146-example teacher dataset is ~a few hundred steps (3 epochs, effective
batch 16), roughly **1–3 h on a free T4/P100**. It finishes comfortably inside
Kaggle's free GPU budget.

**Before you Run All — just two clicks:**
- **Enable the GPU:** *Notebook → Settings → Accelerator → GPU T4 x2* (one T4 is enough).
- *(Optional, only for the push cell)* add your HuggingFace token under
  *Add-ons → Secrets* as `HF_TOKEN`.

That's it — **Run All** trains a real Go specialist on the teacher-distilled
dataset. No dataset to build or upload.

> **Alternative modes** (advanced, in the config cell): a fast `smoke` pipeline
> test on a tiny offline-synthetic sample, or an `upload` mode for your own forge
> dataset added as a Kaggle Dataset. See `TRAINING.md`.

## 1. Configuration

All knobs live here. The **default `MODE = "teacher"`** is a real run — just Run
All. The other modes are alternatives you can flip to:

| `MODE` | Base model | Dataset | Cost / speed |
| --- | --- | --- | --- |
| `"teacher"` *(default)* | `Qwen2.5-Coder-7B-Instruct` | committed **teacher dataset** (146 compile-verified) read from the cloned repo | free on T4/P100, ~1–3 h — **real run** |
| `"smoke"` | `Qwen2.5-Coder-3B-Instruct` | committed **offline-synthetic sample** | free, ~5–15 min — **pipeline smoke test** |
| `"upload"` | `Qwen2.5-Coder-7B-Instruct` | your own **forge dataset** uploaded as a Kaggle Dataset | free on T4/P100 (slow) — **advanced** |

> **Default (`teacher`) run on a free T4/P100:** the 7B 4-bit base fits with
> `seq_len ≤ 1024`, `batch 1`, high grad-accum (set automatically below). The
> teacher dataset is already committed in `guild-code` — **no Kaggle Dataset
> upload needed**. A 24 GB+ card (RTX 3090/4090, A10, L4, A100) trains the 7B
> faster and comfier.

In [ ]:
# ============================================================================
# CONFIG SWITCH — choose ONE mode. The default is a REAL run; just Run All.
#   MODE = "teacher" (DEFAULT) -> REAL run: Qwen2.5-Coder-7B QLoRA on the
#       committed teacher-distilled dataset (146 compile-verified Go examples),
#       read straight from the cloned guild-code repo. NO dataset upload needed.
#       Fits a free T4/P100 at seq<=1024, batch 1, high accum; ~1-3 h.
#   MODE = "smoke" -> fast pipeline smoke test: 3B coder + committed
#       offline-synthetic sample (deterministic placeholders), ~5-15 min.
#   MODE = "upload" -> ADVANCED: 7B coder + a REAL forge dataset you built &
#       uploaded as a Kaggle Dataset (see guild-code DATASETS.md).
# ============================================================================
MODE = "teacher"

if MODE == "teacher":
    # ---- DEFAULT real run: 7B coder + committed teacher dataset ------------
    BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
    REAL_TRAIN_PATH = None   # resolved from the cloned guild-code repo below
    REAL_VAL_PATH   = None
    SEQ_LEN     = 1024   # keep <=1024 so the 7B 4-bit base fits a 16 GB T4
    BATCH_SIZE  = 1      # per-device micro-batch
    GRAD_ACCUM  = 16     # effective batch 16; raise on a 24 GB+ card
    EPOCHS      = 3      # small dataset -> a few hundred steps total
    OUTPUT_DIR  = "/kaggle/working/go_reviewer_adapter"
elif MODE == "upload":
    # ---- ADVANCED: 7B coder + your own uploaded forge dataset --------------
    BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
    # Point these at a REAL forge dataset. On Kaggle, upload the JSONL as a
    # Dataset and it appears under /kaggle/input/<your-dataset>/...
    REAL_TRAIN_PATH = "/kaggle/input/go-code-guild-real/go_code_guild_real_v1.train.jsonl"
    REAL_VAL_PATH   = "/kaggle/input/go-code-guild-real/go_code_guild_real_v1.validation.jsonl"
    SEQ_LEN     = 1024   # keep <=1024 so the 7B 4-bit base fits a 16 GB T4
    BATCH_SIZE  = 1
    GRAD_ACCUM  = 16
    EPOCHS      = 3
    OUTPUT_DIR  = "/kaggle/working/go_reviewer_real_adapter"
elif MODE == "smoke":
    # ---- Pipeline smoke test: 3B coder + committed offline sample ----------
    BASE_MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"
    REAL_TRAIN_PATH = None   # smoke run uses the committed sample (resolved below)
    REAL_VAL_PATH   = None
    SEQ_LEN     = 1024   # short sequences = the biggest VRAM saving
    BATCH_SIZE  = 1
    GRAD_ACCUM  = 8      # effective batch = BATCH_SIZE * GRAD_ACCUM = 8
    EPOCHS      = 2      # 1-2 epochs is plenty for the tiny sample dataset
    OUTPUT_DIR  = "/kaggle/working/go_reviewer_smoke_adapter"
else:
    raise ValueError(f"Unknown MODE: {MODE!r} (use 'teacher', 'smoke', or 'upload')")

LEARNING_RATE = 2.0e-4

# ---- Repos --------------------------------------------------------------
ANVIL_REPO      = "https://github.com/guildlm/anvil.git"
GUILD_CODE_REPO = "https://github.com/guildlm/guild-code.git"

print("MODE      :", MODE)
print("Base model:", BASE_MODEL)
print("Adapter will be saved to:", OUTPUT_DIR)


## 2. Install anvil's training stack + clone the recipe & dataset

We **clone anvil** (so we get both the `src` package *and* the `configs/`
building blocks the recipe references) and editable-install its `[train]` extra
(torch, transformers, peft, trl, bitsandbytes, datasets, huggingface_hub).
We also clone **guild-code** for the `go_reviewer` recipe and the committed Go
dataset.

> **Kaggle note:** Kaggle ships a torch+torchvision pair; we remove torchvision/torchaudio (unused for LLM training) to avoid a `torchvision::nms` version clash.


In [ ]:
import os, sys, subprocess, pathlib

WORK = pathlib.Path("/kaggle/working")
if not WORK.is_dir():           # not on Kaggle? fall back to the current dir
    WORK = pathlib.Path.cwd()

# If we are already *inside* the anvil repo (local dev), use it in place;
# otherwise clone it next to guild-code.
here = pathlib.Path.cwd()
if (here / "src" / "train.py").is_file() and (here / "configs").is_dir():
    ANVIL_DIR = here
    print("Running inside the anvil repo:", ANVIL_DIR)
else:
    ANVIL_DIR = WORK / "anvil"
    if not ANVIL_DIR.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", ANVIL_REPO, str(ANVIL_DIR)], check=True)

GUILD_CODE_DIR = WORK / "guild-code"
if not GUILD_CODE_DIR.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", GUILD_CODE_REPO, str(GUILD_CODE_DIR)], check=True)

print("anvil      ->", ANVIL_DIR)
print("guild-code ->", GUILD_CODE_DIR)


In [ ]:
# Install the training stack from the cloned anvil repo (editable).
# This is the path that ALSO gives us configs/ for recipe reference resolution.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ANVIL_DIR}[train]"], check=True)

# Kaggle ships a matched torch+torchvision pair. Installing anvil[train] can
# bump torch, leaving Kaggle's preinstalled torchvision/torchaudio mismatched.
# transformers then tries to import torchvision and crashes with
#   RuntimeError: operator torchvision::nms does not exist
# Text-LLM SFT does NOT need torchvision/torchaudio, so remove them outright.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"],
    check=False,
)

# --- Alternative (package only, no configs) -----------------------------
# If you only want the library and will supply configs yourself, you can instead:
#   !pip install -q "git+https://github.com/guildlm/anvil.git#egg=guildlm-anvil[train]"
# (We still need the cloned anvil/configs above for the recipe references.)

# Make `import src` resolve to the cloned anvil repo.
if str(ANVIL_DIR) not in sys.path:
    sys.path.insert(0, str(ANVIL_DIR))

print("anvil[train] installed (torchvision/torchaudio removed to avoid version clash).")

## 3. Resolve paths: recipe, configs root, and the committed dataset

The `go_reviewer` recipe references the `qwen2.5_7b` base block and the
`high_rank` LoRA block; those resolve against **anvil's** `configs/` root. In the
default `teacher` mode the dataset is the committed `code_guild_teacher_v1` JSONL
(146 compile-verified examples) read straight from the cloned `guild-code` repo —
no upload needed.

In [ ]:
RECIPE_PATH   = str(GUILD_CODE_DIR / "go" / "anvil" / "go_reviewer.yaml")
CONFIGS_ROOT  = str(ANVIL_DIR / "configs")

if MODE == "teacher":
    # DEFAULT: the committed teacher-distilled dataset (146 compile-verified Go
    # examples), read DIRECTLY from the cloned guild-code repo. No upload.
    DATA_DIR   = GUILD_CODE_DIR / "go" / "datasets" / "code_guild_teacher_v1"
    TRAIN_PATH = str(DATA_DIR / "code_guild_teacher_v1.train.jsonl")
    VAL_PATH   = str(DATA_DIR / "code_guild_teacher_v1.validation.jsonl")
    for p in (RECIPE_PATH, TRAIN_PATH, VAL_PATH):
        assert pathlib.Path(p).is_file(), f"missing: {p}"
elif MODE == "upload":
    # ADVANCED: a dataset YOU built with forge (see guild-code/DATASETS.md) and
    # uploaded as a Kaggle Dataset under /kaggle/input/...
    TRAIN_PATH = REAL_TRAIN_PATH
    VAL_PATH   = REAL_VAL_PATH
    assert pathlib.Path(RECIPE_PATH).is_file(), f"missing recipe: {RECIPE_PATH}"
    assert TRAIN_PATH and pathlib.Path(TRAIN_PATH).is_file(), (
        f"MODE='upload' needs a real dataset at {TRAIN_PATH!r}. Build it with forge "
        "(guild-code/DATASETS.md) and add it as a Kaggle Dataset."
    )
    if VAL_PATH and not pathlib.Path(VAL_PATH).is_file():
        VAL_PATH = None   # optional; anvil can carve a val split from train
else:  # smoke
    # The committed offline-synthetic sample shipped in guild-code.
    DATA_DIR   = GUILD_CODE_DIR / "go" / "datasets" / "code_guild_sample_v1"
    TRAIN_PATH = str(DATA_DIR / "code_guild_sample_v1.train.jsonl")
    VAL_PATH   = str(DATA_DIR / "code_guild_sample_v1.validation.jsonl")
    for p in (RECIPE_PATH, TRAIN_PATH, VAL_PATH):
        assert pathlib.Path(p).is_file(), f"missing: {p}"

print("MODE   :", MODE)
print("recipe :", RECIPE_PATH)
print("configs:", CONFIGS_ROOT)
print("train  :", TRAIN_PATH)
print("val    :", VAL_PATH)


### Peek at the dataset with anvil's **real** `src.data` loader

`src.data.format_example` is the exact pure-python formatter anvil uses during
training. We render one row with the recipe's system prompt so you can see what
the model will actually be trained on.


In [ ]:
import json
from src.data import format_example

with open(TRAIN_PATH, "r", encoding="utf-8") as fh:
    first_row = json.loads(fh.readline())

print("Row schema keys:", sorted(first_row))
print("\n--- formatted training text (ChatML fallback preview) ---\n")
print(format_example(
    first_row,
    system_prompt="You are a meticulous senior Go engineer performing code review.",
)[:1200])


## 4. Train — anvil's real QLoRA SFT entrypoint

We load the recipe into anvil's real `AnvilConfig` via `load_recipe`, override a
handful of fields for the **free T4** (short sequences, grad accumulation,
**fp16** because Turing T4s have no bf16, point at the committed teacher
dataset), then hand it straight to **`src.train.train()`** — the same function
the `anvil-train` CLI calls. `train()` loads the base in 4-bit (QLoRA), applies a
single LoRA via TRL's `SFTTrainer`, and saves the adapter.

In [ ]:
from src.config import load_recipe
from src.train import train

# 1) Load the committed recipe (resolves base_model/lora refs vs anvil/configs).
recipe = load_recipe(RECIPE_PATH, configs_root=CONFIGS_ROOT)

# 2) Override for this run (teacher 7B + committed dataset, smoke 3B sample,
#    or upload 7B + your own dataset).
recipe.base_model.model_id            = BASE_MODEL
recipe.base_model.attn_implementation = None   # T4/P100 have no FlashAttention-2
recipe.dataset.path        = TRAIN_PATH
recipe.dataset.eval_path   = VAL_PATH
recipe.dataset.val_split   = 0.0 if VAL_PATH else 0.05  # split from train if no val file
recipe.dataset.max_seq_length = SEQ_LEN
recipe.output_dir          = OUTPUT_DIR

recipe.sft.batch_size                  = BATCH_SIZE
recipe.sft.gradient_accumulation_steps = GRAD_ACCUM
recipe.sft.epochs                      = EPOCHS
recipe.sft.learning_rate               = LEARNING_RATE
recipe.sft.bf16 = False                         # Turing T4 / Pascal P100: use fp16
recipe.sft.fp16 = True

print("MODE                :", MODE)
print("Effective seq length:", recipe.effective_max_seq_length)
print("Base model          :", recipe.base_model.model_id)
print("LoRA r / alpha      :", recipe.lora.r, "/", recipe.lora.alpha)
print("Output dir          :", recipe.output_dir)


In [ ]:
# This is the heavy step (loads the 4-bit base, runs SFT).
# Default `teacher` run: ~1-3 h on a free T4/P100 (7B QLoRA, 146 examples,
# 3 epochs -> a few hundred steps). The `smoke` mode finishes in ~5-15 min.
adapter_dir = train(recipe)
print("\nTrained LoRA adapter saved to:", adapter_dir)


In [ ]:
# Confirm the adapter files landed.
import os
print("Contents of", OUTPUT_DIR, ":")
for name in sorted(os.listdir(OUTPUT_DIR)):
    print("  ", name)


## 5. Sanity check — generate a Go review with the trained adapter

Load the 4-bit base + the LoRA adapter and generate a short review for a small
Go snippet. In the default `teacher` mode the adapter was trained on real,
compile-verified teacher reviews, so the output should be substantive. (In
`smoke` mode the *content* is placeholder-flavoured — that mode only verifies the
**plumbing**: base + adapter load and generate.)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto",
)
model = PeftModel.from_pretrained(base, OUTPUT_DIR)
model.eval()

snippet = """package main

func Divide(a, b int) int {
    return a / b   // no zero check
}"""
messages = [
    {"role": "system", "content": "You are a meticulous senior Go engineer performing code review."},
    {"role": "user", "content": "Review this Go code:\n\n" + snippet},
]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))


## 6. *(Optional)* Push the adapter to the HuggingFace Hub — free hosting

This uses anvil's `anvil-push` helper (`src.hub.push_adapter`), which writes a
minimal model card and uploads the adapter folder. **Hosting on the Hub is free.**

**Setup:** add your HF token under *Add-ons → Secrets* as `HF_TOKEN`
(or paste it below). Create the token at https://huggingface.co/settings/tokens
with **write** access. Then set `HF_REPO_ID` and run the cell.


In [ ]:
# ---- EDIT THIS, then set PUSH = True to upload --------------------------
HF_REPO_ID = "your-username/go-reviewer-lora"   # <-- change me
PUSH = False                                    # <-- set True to actually push
PRIVATE = False
# ------------------------------------------------------------------------

if PUSH:
    import os
    # Prefer Kaggle Secrets, then env var.
    token = os.environ.get("HF_TOKEN")
    try:
        from kaggle_secrets import UserSecretsClient
        token = token or UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

    from src.hub import push_adapter
    url = push_adapter(
        HF_REPO_ID, OUTPUT_DIR, token=token, private=PRIVATE, base_model=BASE_MODEL,
    )
    print("Pushed:", url)
else:
    print("PUSH is False — skipping. Set PUSH = True (and HF_REPO_ID) to upload.")
    print("CLI equivalent:")
    print(f"  anvil-push --repo-id {HF_REPO_ID} --adapter {OUTPUT_DIR} --base-model {BASE_MODEL}")


## Next steps

- **Merge + GGUF for local serving:** `anvil-merge` then `anvil-quantize --method gguf`,
  then run in **Ollama**. See [`TRAINING.md`](https://github.com/guildlm/anvil/blob/main/TRAINING.md).
- **Scale further:** the default already trains the real 7B coder on the teacher
  dataset for $0. To train faster (or push more epochs / longer context), rent a
  24 GB+ GPU (Vast.ai / RunPod RTX 4090, ~$1–2) — see `TRAINING.md`.
- **Grow the dataset:** add more teacher-distilled examples in `guild-code` and
  re-run, or flip `MODE = "upload"` to train on your own forge dataset.